In [1]:
# ==============================================================================
# Continuous Structured Streaming pipeline from Bronze to Silver telemetry
# ==============================================================================

from pyspark.sql.functions import col, to_timestamp, coalesce, lit, when, upper, trim
from pyspark.sql.types import IntegerType, DoubleType

# ------------------------------------------------------------------------------
# 1. READ CONTINUOUS STREAM FROM BRONZE DELTA TABLE
# ------------------------------------------------------------------------------
# Reads raw JSON telemetry records streaming in from Eventstream
df_bronze_stream = (
    spark.readStream
    .format("delta")
    .table("dbo.bronze_live_telemetry")
)

# ------------------------------------------------------------------------------
# 2. TRANSFORM, CLEAN, AND ENFORCE SCHEMA FOR SILVER
# ------------------------------------------------------------------------------
df_silver_transformed = (
    df_bronze_stream
    
    # Timestamps & Core Keys
    .withColumn("event_timestamp", to_timestamp(col("timestamp")))
    .withColumn("year", col("event_timestamp").cast("date").cast("string").substr(1, 4).cast("integer"))
    .withColumn("month", col("event_timestamp").cast("date").cast("string").substr(6, 2).cast("integer"))
    .withColumn("day", col("event_timestamp").cast("date").cast("string").substr(9, 2).cast("integer"))
    .withColumn("flight_number", col("flight_number").cast(IntegerType()))
    
    # Categorical Dimension Keys (Clean & Trim)
    .withColumn("airline_code", upper(trim(col("airline"))))
    .withColumn("origin_airport_code", upper(trim(col("origin"))))
    .withColumn("dest_airport_code", upper(trim(col("destination"))))
    
    # Flight Position & Telemetry Measures
    .withColumn("current_altitude_ft", coalesce(col("current_altitude_ft").cast(IntegerType()), lit(0)))
    .withColumn("speed_knots", coalesce(col("speed_knots").cast(IntegerType()), lit(0)))
    
    # Operational Delays (Coalesce missing values to 0)
    .withColumn("departure_delay_mins", coalesce(col("departure_delay_mins").cast(IntegerType()), lit(0)))
    .withColumn("arrival_delay_mins", lit(0).cast(IntegerType()))  # Live stream defaults to 0 until arrival
    
    # Calculated Status Flags
    .withColumn("is_delayed_flag", when(col("departure_delay_mins") > 15, 1).otherwise(0))
    .withColumn("is_cancelled_flag", lit(0).cast(IntegerType()))
    .withColumn("is_diverted_flag", lit(0).cast(IntegerType()))
    
    # Schema Padding for Batch Compatibility (Pad missing batch fields with NULL/0)
    .withColumn("cancellation_code", lit(None).cast("string"))
    .withColumn("air_system_delay_mins", lit(0).cast(IntegerType()))
    .withColumn("security_delay_mins", lit(0).cast(IntegerType()))
    .withColumn("airline_delay_mins", lit(0).cast(IntegerType()))
    .withColumn("late_aircraft_delay_mins", lit(0).cast(IntegerType()))
    .withColumn("weather_delay_mins", lit(0).cast(IntegerType()))
    
    # Metadata Audit Tracking
    .withColumn("record_source", lit("EVENTSTREAM_LIVE_TELEMETRY"))
    .withColumn("silver_ingestion_timestamp", lit(to_timestamp(lit(None)))) # Updated at write
)

# ------------------------------------------------------------------------------
# 3. WRITE CONTINUOUS STREAM TO SILVER DELTA TABLE WITH CHECKPOINTING
# ------------------------------------------------------------------------------
checkpoint_path = "Files/checkpoints/silver_live_telemetry"

query = (
    df_silver_transformed.writeStream
    .format("delta")
    .outputMode("append")
    .option("checkpointLocation", checkpoint_path)
    .option("mergeSchema", "true")
    .toTable("dbo.silver_live_telemetry")
)

print("Continuous stream started: dbo.bronze_live_telemetry ---> dbo.silver_live_telemetry")

StatementMeta(, 358a150a-da45-42b3-bc4e-898648ed7e21, 3, Finished, Available, Finished, False)

Continuous stream started: dbo.bronze_live_telemetry ---> dbo.silver_live_telemetry
